In [304]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [305]:
df = pd.read_csv('orbis_edges_0514.csv')
df2 = pd.read_csv('orbis_nodes_0514.csv')
print(df.isnull().sum())
print("*")
print(df2.isnull().sum())
print("*")
print(df.shape)
print(df2.shape)

source     0
target     0
km         0
days       0
expense    0
type       0
dtype: int64
*
id        0
label     0
x        55
y        55
dtype: int64
*
(2208, 6)
(678, 4)


In [306]:
print(df2.dtypes)
df2.isna().any()[lambda x: x]
df2 [ df2['x'].notnull() ] ['label']

id         int64
label     object
x        float64
y        float64
dtype: object


0        Ad fl. Tigrim
1      Iulia Concordia
2                 Roma
3           Praetorium
4             Patavium
            ...       
673              Syros
674              Thera
675              Tenos
676              Telos
677            Isthmia
Name: label, Length: 623, dtype: object

In [307]:
fig = px.scatter_geo(data_frame=df2,lat='x',lon='y',hover_name='label')
fig.update_geos(lataxis_range =[22,60], lonaxis_range=[-10,45], resolution = 50)
fig.show()

In [308]:
df2_clean = df2 [ df2['x'].notnull() ]
df2_clean = df2_clean[~((df2_clean.x ==0) & (df2_clean.y ==0))]
df2_clean

,id,label,x,y
0,50002,Ad fl. Tigrim,37.325000,42.196000
1,50209,Iulia Concordia,45.757000,12.844000
2,50327,Roma,41.892000,12.486000
3,50319,Praetorium,31.500000,15.500000
4,50294,Patavium,45.410000,11.877000
...,...,...,...,...
673,50798,Syros,37.430000,24.920000
674,50801,Thera,36.416667,25.433333
675,50800,Tenos,37.553000,25.116000
676,50799,Telos,36.433000,27.336000


In [309]:
fig = px.scatter_geo(data_frame=df2_clean,lat='x',lon='y',hover_name='label')
fig.update_geos(lataxis_range =[22,60], lonaxis_range=[-10,45], resolution = 50)
fig.show()

In [310]:
df_clean = df[df['target'].isin(df2_clean['id'])&df['source'].isin(df2_clean['id'])]
df_clean.isnull().sum()
df2_clean.isnull().sum()

id       0
label    0
x        0
y        0
dtype: int64

In [311]:
G = nx.from_pandas_edgelist( df_clean, source='source', target='target', edge_attr=["km","days","expense","type"])
print(nx.is_connected(G))
print(nx.number_connected_components(G))

False
5


In [312]:
clean= pd.merge(df_clean, df2_clean, left_on='source', right_on='id')
clean= pd.merge(clean, df2_clean, left_on='target', right_on='id', suffixes=('_source', '_target'))
edges_geo = clean

In [314]:
node_map = go.Figure()
lats = []
lons = []
for index, row in edges_geo.iterrows():
    lats.append(row['x_source'])
    lats.append(row['x_target'])
    lats.append(None)
    lons.append(row['y_source'])
    lons.append(row['y_target'])
    lons.append(None)

node_map.add_trace(go.Scattergeo(lat=lats,lon=lons,mode='lines',line=dict(width=0.5, color='blue')))
node_map.update_geos(lataxis_range =[22,60], lonaxis_range=[-10,45], resolution = 50)
node_map.show()